In [0]:
spark

In [0]:
df = spark.read.format("csv").option("header", "true").load("/Volumes/workspace/default/sample_data/Employee.csv")

In [0]:
df.show()

In [0]:
df.schema

In [0]:
from pyspark.sql.types import StringType, StructField, StructType, DateType, IntegerType

schema = StructType(
    [
        StructField("Education", StringType(), True),
        StructField("JoiningYear", DateType(), True),
        StructField("City", StringType(), True),
        StructField("PaymentTier", IntegerType(), True),
        StructField("Age", IntegerType(), True),
        StructField("Gender", StringType(), True),
        StructField("EverBenched", StringType(), True),
        StructField("ExperienceInCurrentDomain", IntegerType(), True),
        StructField("LeaveOrNot", IntegerType(), True),
    ]
)

options = {
    "header" : True
}

df = spark.read.format("csv").schema(schema).options(**options).load("/Volumes/workspace/default/sample_data/Employee.csv")

In [0]:
df.printSchema()

In [0]:
df.show(5)

In [0]:
from pyspark.sql.functions import col, expr

df.select(df.Age, expr("City")).show(5)

In [0]:
df.selectExpr("Age", "City").show(5)

In [0]:
df.select(col("Age"), col("City")).show(5)

In [0]:
from pyspark.sql.functions import year

df_1 = df.withColumn("Year", year(col("JoiningYear")))
df_1.show(5)

In [0]:
df_1.printSchema()

In [0]:
df2 = df_1.withColumnsRenamed({"JoiningYear": "JoiningDate", "Year": "JoiningYear"})
df2.show(5)

In [0]:
from pyspark.sql.functions import when

df2 = df2.withColumn("LeaveOrNot", when(col("LeaveOrNot") == 1, 
                                        "Yes").when(col("LeaveOrNot") == 0, 
                                                    "No").otherwise(None))
df2.show()

In [0]:
df2.filter(df2.LeaveOrNot.isNull()).count()

In [0]:
df2.where(col("JoiningYear") > 2017).show()

In [0]:
from pyspark.sql.functions import min, max

df2.select(min("JoiningYear").alias("min_year"), max("JoiningYear").alias("max_year")).show()

In [0]:
from pyspark.sql.functions import lit

df2 = df2.withColumn("Current_Year", lit(2026).cast("int"))
df2.show(5)

In [0]:
df2 = df2.withColumn("New_Experience", when(
    col("ExperienceInCurrentDomain") > 2, "Experienced").when(
    col("ExperienceInCurrentDomain") <= 2, "Newbie").otherwise(None
))

df2.show(5)

In [0]:
from pyspark.sql.functions import regexp_replace

rege_m = regexp_replace(col("Gender"), "Male", "M")
rege_f = regexp_replace(col("Gender"), "Female", "F")


df2 = df2.withColumn("Gender", rege_m).withColumn("Gender", rege_f)
df2.show(5)

In [0]:
from pyspark.sql.functions import to_date, date_format

df2 = df2.withColumn("JoiningDate", date_format(col("JoiningDate"), "dd-MM-yyyy"))
df2.show(5)

In [0]:
df2 = df2.sort(col("Age").asc())
df2.show(5)

In [0]:
from pyspark.sql.functions import current_date, current_timestamp

df2 = df2.withColumn("Current_Date", current_date()).withColumn("Current_Timestamp", current_timestamp())
display(df2)

In [0]:
from pyspark.sql.functions import count

df3 = df2.groupBy("Education").agg(count("*").alias("total_emps")).where("total_emps > 500")
df3.show()

In [0]:
df2.distinct().count()

In [0]:
df4 = df2.dropDuplicates()
display(df4)

In [0]:
df2.select(col("Education")).distinct().show()

In [0]:
df4.createOrReplaceTempView("df4")

In [0]:
%sql
select df4.*, 
      row_number() over(partition by `City` order by `JoiningYear`) as rn
from df4
-- order by df4.`JoiningYear`

In [0]:
from pyspark.sql.functions import rand

df5 = df4.withColumn("Salary", (rand() * (150000 - 10000) + 10000).cast("int"))
df5.display()

In [0]:
from pyspark.sql.functions import floor, col

df5 = df5.withColumn("Rounded_Salary", floor(col("Salary") / 1000) * 1000)
df5.display()

In [0]:
df5.createOrReplaceTempView("df5")

In [0]:
%sql
select Rounded_Salary, count(*)
from df5
group by Rounded_Salary

In [0]:
%sql
select 
  Salary,
  Rounded_Salary,
  rank() over(order by Rounded_Salary asc) as Rank_Salary,
  dense_rank() over(order by Rounded_Salary asc) as Dense_Salary,
  row_number() over(partition by Rounded_Salary order by Rounded_Salary asc) as Row_Salary,
from df5